In [5]:
"""Percolation functions.

Notes:
- create `images/` (or change file paths below)
- cd to the directory containing this file and run `python3 percolation.py`
OR import and run in .ipynb
- add/change functions in line 96 to run what you want when calling the script
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

##############################################
# adapted for POCs 1

L_VALUES = [20, 50, 100, 200, 500, 1000]  # square lattice of size L
# Pc for square lattices is a known/accepted quantity obtained from simulations
# Visually, we can see it is approx 0.59, and wikipedia says it is 0.59274621
# https://en.wikipedia.org/wiki/Percolation_theory
PERC_CRIT_PROB = 0.59274621  # p_c


def percolation_model(prob, L):
    """Run site percolation, return world as 2D array (the 'world')."""
    return np.random.choice([0, 1], size=(L, L), p=[1-prob, prob])


def get_perc_largest_cluster(prob, L, N_sim=0):
    if N_sim > 0:
        all_cluster_sizes = []

    # Run multiple simulations to get average cluster sizes
        for _ in range(N_sim):
            world = percolation_model(prob, L)
            structure = [[0, 1, 0], [1, 1, 1], [0, 1, 0]]
            label_world, nb_labels = ndimage.label(world, structure)
            cluster_sizes = ndimage.sum(world, label_world, range(nb_labels + 1))
            all_cluster_sizes.extend(cluster_sizes)

        return np.max(all_cluster_sizes), all_cluster_sizes
    """Get the largest cluster in a percolation model."""
    world = percolation_model(prob, L)  # run model

    # filter largest cluster
    # Site percolation
    structure = [[0, 1, 0], [1, 1, 1], [0, 1, 0]]  # define connection
    # label_world is array of same shape as world, with each elem (cluster) labeled
    # nb_labels is the number of labels/clusters
    label_world, nb_labels = ndimage.label(world, structure)
    # ndimage.sum sums values in world for each label in label_world
    # range(nb_labels + 1) says labels to sum over
    # cluster_sizes is array of elems, each is sum of values in world for that label
    cluster_sizes = ndimage.sum(world, label_world, range(nb_labels + 1))
    return cluster_sizes.max(), cluster_sizes


# Parameters
def percolation_simulation():
    """Run entire percolation simulation."""
    probabilities = np.linspace(0, 1, 100)  # occupation probs (increments of 10^-2)

    # For each L, run N_tests for occupation prob
    for L in L_VALUES:
        print(f"L = {L}")
        prob_to_max_avg_cluster = []
        for p in probabilities:
            # run N_tests=100 of the percolation model
            # (a loop would be equivalent, but we're optimizing below
            max_cluster_sizes = [get_perc_largest_cluster(p, L)[0] for _ in range(100)]
            max_avg_cluster_size = np.mean(max_cluster_sizes)
            # fractional size: normalize the avg size by the total number of cells (L)
            prob_to_max_avg_cluster.append(max_avg_cluster_size / (L * L))
        plt.plot(probabilities, prob_to_max_avg_cluster, label=f"L = {L}")

    plt.axvline(PERC_CRIT_PROB, color="black", linestyle="--",
                label="Critical probability")
    plt.legend()
    plt.xlabel("p")
    plt.ylabel("S avg")
    print("Simulation complete and plot saved.")


def cluster_dist_plot(L=1000):
    dict_p_c_vals = {
        PERC_CRIT_PROB: 0,  # at critical probability
        PERC_CRIT_PROB / 2: 1,  # below critical
        PERC_CRIT_PROB + (1 - PERC_CRIT_PROB) / 2: 2  # above critical
    }
    total_sites = L * L  # Total number of sites in the lattice

    for p_c, label in dict_p_c_vals.items():
        _, all_cluster_sizes = get_perc_largest_cluster(p_c, L, N_sim=100)  # average over 100 simulations
        df = pd.DataFrame(all_cluster_sizes, columns=["clust_size"])
        dist = df['clust_size'].value_counts().sort_index()
        dist = dist.rename_axis('clust_size').reset_index(name='frequency')

        # Normalize the cluster sizes by total lattice area
        dist["clust_size"] = dist["clust_size"] / total_sites

        # Normalize the frequency by the total number of clusters
        total_clusters = dist["frequency"].sum()
        dist["log_frequency"] = dist["frequency"] / total_clusters  # Fraction of clusters
        dist["log_frequency"] = np.log10(dist["log_frequency"]) 

        plt.clf()
        plt.scatter(dist["clust_size"], dist["log_frequency"], marker="o", s=10)

        plt.xscale("log")

        plt.xlim([10**-6, 1]) 

        plt.ylim([-8, 0])

        plt.xlabel("log10 S")
        plt.ylabel("log10 P(S)")
        plt.title(f"Cluster Size Distribution for p = {p_c}")

        # plt.savefig(f"images/cluster_size_dist_{label}_{L}.png")
        plt.show()

# 3
percolation_simulation()
# 4
# cluster_dist_plot()

L = 20
L = 50
L = 100
L = 200
L = 500
